# Experiment 1 - Getting Started with Python, TensorFlow, and Hugging Face/OpenAI

**Aim:** Set up a Generative AI environment, inspect the runtime, generate cricket-coaching answers from five prompts, and compare generation settings.

This notebook keeps the original experiment sections while using a safe OpenAI environment-variable integration. It never stores an API key in the notebook.

## Data source and reproducibility

The BattingCoach project documents a Roboflow cricket-bat dataset, YOLOv8-OBB bat detection, MediaPipe pose landmarks, and a ten-class VideoMAE shot taxonomy. The image dataset is not used to train this language experiment; its documented domain provides the prompt context.

## 1. Install and verify dependencies

Dependencies are installed outside the notebook because the selected Python environment is externally managed. This cell only reports versions.

In [1]:
from importlib.metadata import version
for package in ("openai", "pandas", "torch", "transformers"):
    try:
        print(f"{package}: {version(package)}")
    except Exception as exc:
        print(f"{package}: unavailable ({exc})")


openai: 3.42.1
pandas: 3.1.0
torch: 2.15.0
transformers: 5.22.1


## 2. Check the Python environment

In [2]:
import sys, platform
try:
    import torch
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
except ImportError:
    torch_version = "unavailable"
    cuda_available = False
print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch_version)
print("CUDA available:", cuda_available)
if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.14.5 (main, May 10 2026, 20:29:46) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26300-SP0
PyTorch: 2.15.0+cpu
CUDA available: False


## 3. Configure the text-generation model

The API key is read only from `OPENAI_API_KEY`. If it is absent, the notebook uses a clearly labelled offline preview so every cell remains executable.

In [3]:
import os
import pandas as pd
MODEL_ID = "gpt-6-luna"
api_key = os.getenv("OPENAI_API_KEY")
USE_OPENAI = bool(api_key)
if USE_OPENAI:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
print("Model:", MODEL_ID)
print("Execution mode:", "OpenAI API" if USE_OPENAI else "Offline preview")


Model: gpt-6-luna
Execution mode: Offline preview


## 4. Define five BattingCoach prompts

In [4]:
prompts = [
    "Explain what a cricket cover drive is and give two beginner technique checks.",
    "Give three concise checks a batting coach should use to assess a batter's balance.",
    "Explain how BatCoach can analyse batting technique using pose landmarks, bat detection and shot classification.",
    "Give two safe beginner cues for playing a straight drive.",
    "Before playing a pull shot, list three things the batter should check."
]
for i, prompt in enumerate(prompts, start=1):
    print(f"{i}. {prompt}")


1. Explain what a cricket cover drive is and give two beginner technique checks.
2. Give three concise checks a batting coach should use to assess a batter's balance.
3. Explain how BatCoach can analyse batting technique using pose landmarks, bat detection and shot classification.
4. Give two safe beginner cues for playing a straight drive.
5. Before playing a pull shot, list three things the batter should check.


## 5. Generate and display all five answers

In [5]:
preview_answers = [
    "A cover drive is a front-foot shot through the off-side cover region. Keep the head still over the front knee and use a controlled vertical bat face.",
    "Check that the head stays level, the feet remain stable at contact, and the batter finishes under control without falling away.",
    "BatCoach can combine MediaPipe pose landmarks, YOLOv8-OBB bat detection and VideoMAE stroke classification to analyse technique.",
    "Move the front foot towards the ball, keep the head over the front knee, and swing with a straight bat and relaxed follow-through.",
    "Check that the ball is short enough, transfer weight onto the back foot with a still head, and control the horizontal swing toward the leg side."
]
def generate_answer(prompt, preview_answer):
    if not USE_OPENAI:
        return preview_answer
    response = client.responses.create(
        model=MODEL_ID,
        input=[
            {"role":"system", "content":"You are a concise, accurate cricket batting coach. Answer only the question. Do not invent project facts or give medical diagnoses."},
            {"role":"user", "content":prompt}
        ],
        max_output_tokens=96
    )
    answer = response.output_text.strip()
    if not answer:
        raise RuntimeError(f"The API returned an empty answer for prompt {prompt!r}")
    return " ".join(answer.split())
results = pd.DataFrame([
    {"Prompt #": i, "Prompt": prompt, "Answer": generate_answer(prompt, preview)}
    for i, (prompt, preview) in enumerate(zip(prompts, preview_answers), start=1)
])
results


,Prompt #,Prompt,Answer
0,1,Explain what a cricket cover drive is and give...,A cover drive is a front-foot shot through the...
1,2,Give three concise checks a batting coach shou...,"Check that the head stays level, the feet rema..."
2,3,Explain how BatCoach can analyse batting techn...,"BatCoach can combine MediaPipe pose landmarks,..."
3,4,Give two safe beginner cues for playing a stra...,"Move the front foot towards the ball, keep the..."
4,5,"Before playing a pull shot, list three things ...","Check that the ball is short enough, transfer ..."


## 6. Compare generation parameters

In [6]:
parameter_comparison = pd.DataFrame([
    ["Focused", 0.2, 96, "Recommended: concise and consistent."],
    ["Balanced", 0.7, 128, "More wording variation and more token use."],
    ["Creative", 1.0, 160, "Highest variation; greater consistency risk."],
], columns=["Setting", "Temperature", "Max output tokens", "Expected behaviour"])
parameter_comparison


,Setting,Temperature,Max output tokens,Expected behaviour
0,Focused,0.2,96,Recommended: concise and consistent.
1,Balanced,0.7,128,More wording variation and more token use.
2,Creative,1.0,160,Highest variation; greater consistency risk.


## 7. BattingCoach project metadata

In [7]:
project_dataset_metadata = {
    "bat_detection_dataset": "Cricket Bat Dataset (Roboflow)",
    "annotated_images": 930,
    "annotation_format": "YOLOv8 Oriented Bounding Box",
    "class_names": ["Bat"],
    "shot_classification_dataset_id": "rokmr/cricketshot",
    "shot_classes": ["cover", "defense", "flick", "hook", "late_cut", "lofted", "pull", "square_cut", "straight", "sweep"]
}
pd.DataFrame([project_dataset_metadata])


,bat_detection_dataset,annotated_images,annotation_format,class_names,shot_classification_dataset_id,shot_classes
0,Cricket Bat Dataset (Roboflow),930,YOLOv8 Oriented Bounding Box,[Bat],rokmr/cricketshot,"[cover, defense, flick, hook, late_cut, lofted..."


## 8. Observations and conclusion

The notebook evaluates five cricket-specific prompts and records every answer in one table. Lower output limits reduce token use. A low temperature is preferred for factual coaching guidance. The offline preview is not an API result; it is a deterministic fallback for execution testing. For real model results, configure a newly rotated key through `OPENAI_API_KEY` before starting the kernel.

Never paste an API key into this notebook or commit it to source control.